In [ ]:
# Example 1: Inspect the top-level package surface.
import jupyterlab_chat

public_names = [n for n in dir(jupyterlab_chat) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for name in public_names[:30]:
    obj = getattr(jupyterlab_chat, name)
    print(f"  {name}: {type(obj).__name__}")

assert hasattr(jupyterlab_chat, '__path__'), 'jupyterlab_chat should be a package'


In [ ]:
# Example 2: Locate the Jupyter Server extension entry points.
import jupyterlab_chat

# Jupyter Server extensions expose _jupyter_server_extension_points
ext_fn = getattr(jupyterlab_chat, '_jupyter_server_extension_points', None)
print(f"_jupyter_server_extension_points: {ext_fn}")

if callable(ext_fn):
    points = ext_fn()
    print(f"Extension points: {points}")
    assert isinstance(points, list)
    for p in points:
        assert 'module' in p
else:
    print("No extension-points hook exposed at top level (may be in a submodule)")


In [ ]:
# Example 3: The package itself is the Jupyter Server extension module; check its standard hooks.
# (measured, jupyterlab-chat 0.25.0: there is no `jupyterlab_chat.extension` or `.app` submodule)
import jupyterlab_chat

points = jupyterlab_chat._jupyter_server_extension_points()
print(f"Server extension points: {points}")
assert points == [{'module': 'jupyterlab_chat'}], points
assert callable(jupyterlab_chat._load_jupyter_server_extension)

lab_paths = jupyterlab_chat._jupyter_labextension_paths()
print(f"Lab extension paths: {lab_paths}")
assert lab_paths == [{'src': 'labextension', 'dest': 'jupyterlab-chat-extension'}], lab_paths


In [ ]:
# Example 4: Check the bundled labextension static assets are findable on disk.
import json
import pathlib

import jupyterlab_chat

pkg_root = pathlib.Path(jupyterlab_chat.__path__[0])
labext_dirs = list(pkg_root.glob('labextension')) + list(pkg_root.glob('*/labextension'))
print(f"labextension dirs: {labext_dirs}")

for d in labext_dirs:
    pj = d / 'package.json'
    if pj.exists():
        meta = json.loads(pj.read_text())
        print(f"  {pj}: name={meta.get('name')} version={meta.get('version')}")
        assert meta.get('name'), 'labextension package.json should declare a name'


In [ ]:
# Example 5: Verify core runtime deps are importable alongside jupyterlab_chat.
import importlib

required = ['jupyter_server', 'jupyter_ydoc', 'pycrdt']
results = {}
for name in required:
    try:
        mod = importlib.import_module(name)
        results[name] = getattr(mod, '__version__', 'unknown')
    except Exception as e:
        results[name] = f"FAILED: {type(e).__name__}: {e}"

for name, info in results.items():
    print(f"  {name}: {info}")

missing = [n for n, info in results.items() if isinstance(info, str) and info.startswith('FAILED')]
assert not missing, f"Missing runtime deps: {missing}"
print("\nAll declared runtime deps import cleanly")
